# 00 · Environment, NCCL, GDN kernels, DDP speedup  (Phase 0, ~0.75 GPU-h)

**Accelerator:** GPU T4 ×2 · **Internet:** on

**What it does**
1. Prints the hardware and software report.
2. Runs an NCCL all-reduce smoke test across the two T4s, with a fallback to `NCCL_P2P_DISABLE=1`.
3. Installs FLA's kernel package `fla-core` (`--no-deps`, so Kaggle's torch is never replaced) and runs **test 4**: FLA's Triton GDN kernel and our pure-PyTorch
   kernel against FLA's naive reference (outputs + gradients, fp32 and fp16), plus fwd+bwd speed at T ∈ {512, 1024, 2048}.
4. Measures 1-GPU vs 2-GPU (DDP) training throughput for A1/A2 at a Phase 1 cell, for each usable backend.

**Outputs:** `results/phase0/E0.kernels/*.json`, `results/phase0/E0.thr/*.json`, and the recommended `GDN_BACKEND`.

**Stop and ask if:** NCCL fails with and without the fallback; or the *torch* kernel FAILs correctness (that is a bug).
An FLA failure is fine: use `GDN_BACKEND = "torch"`.

In [ ]:
# ===================== SETTINGS: edit this cell, then "Run All" (or "Save Version -> Save & Run All") =====================
REPO_URL     = "https://github.com/Shiveshrane/Engram.git"   # None -> copy the code from CODE_DATASET instead
REPO_REF     = "main"           # pin the PREREG commit hash for Phase 1 / 2 runs
CODE_DATASET = "/kaggle/input/engramdelta-src"   # Kaggle Dataset with the repo (used when REPO_URL is None)
HF_REPO      = None             # e.g. "your-hf-user/engramdelta-runs" (private dataset repo; needs Kaggle Secret HF_TOKEN)
GDN_BACKEND  = "torch"          # "torch" (pure PyTorch, always works) or "fla" (only if NB00 says PASS)
FLA_SPEC     = "fla-core==0.5.2"                 # FLA's kernel package (fla.ops); installed with --no-deps
PREV_OUTPUTS = []               # outputs of earlier sessions attached as inputs, e.g. ["/kaggle/input/engramdelta-11-phase1-h1-grid"]
NCCL_ENV     = {}               # set to {"NCCL_P2P_DISABLE": "1", "NCCL_IB_DISABLE": "1"} if NB00's first NCCL test hung

In [ ]:
# ===================== Session setup (no edits needed) =====================
import os, sys, subprocess, shutil, pathlib, json, glob, time
NEEDS_GPU, INSTALL_FLA = True, True
ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = pathlib.Path("/kaggle/working") if ON_KAGGLE else pathlib.Path.cwd()
# Locally (notebook opened from the repo's notebooks/ folder) the repo is the parent directory.
REPO = WORK / "engramdelta" if ON_KAGGLE else (WORK.parent if WORK.name == "notebooks" else WORK)

def run(cmd, check=True, env=None, timeout=None, show=None):
    """Run a shell command in the repo and stream its output into this notebook."""
    print("$", show or cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, **(env or {})}, cwd=str(REPO) if REPO.exists() else str(WORK))
    t0 = time.time()
    try:
        for line in p.stdout:
            print(line, end="", flush=True)
            if timeout and time.time() - t0 > timeout:
                raise subprocess.TimeoutExpired(cmd, timeout)
        rc = p.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        p.kill(); print(f"!! timed out after {timeout}s"); rc = -9
    if check and rc != 0:
        raise RuntimeError(f"exit code {rc}: {show or cmd}")
    return rc

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

# 1. code
if not REPO.exists():
    if REPO_URL:
        url, tok = REPO_URL, secret("GITHUB_TOKEN")
        if tok and url.startswith("https://"):
            url = url.replace("https://", f"https://{tok}@")
        run(f"git clone -q {url} {REPO}", show=f"git clone {REPO_URL}")
        run(f"git -C {REPO} checkout -q {REPO_REF}")
    else:
        shutil.copytree(CODE_DATASET, REPO)
os.chdir(REPO); sys.path.insert(0, str(REPO))
try:
    commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
except Exception:
    commit = open("GIT_COMMIT").read().strip() if os.path.exists("GIT_COMMIT") else "unknown"
os.environ["GIT_COMMIT"] = commit
print("code commit:", commit)

# 2. packages (never let pip replace Kaggle's torch)
run("pip install -q --no-deps einops sympy")
if INSTALL_FLA or GDN_BACKEND == "fla":
    run(f"pip install -q --no-deps {FLA_SPEC}", check=False)

# 3. environment for every launched run
os.environ["ENGRAMDELTA_GDN_BACKEND"] = GDN_BACKEND
os.environ["ENGRAMDELTA_CKPT_DIR"] = str(WORK / "ckpt")
os.environ.update(NCCL_ENV)
if HF_REPO and secret("HF_TOKEN"):
    os.environ["HF_REPO"], os.environ["HF_TOKEN"] = HF_REPO, secret("HF_TOKEN")
    print("HF Hub sync: ON ->", HF_REPO)
else:
    print("HF Hub sync: OFF (results persist via this notebook's output only)")

# 4. restore results / checkpoints from earlier sessions (never overwrites existing files)
def restore(src, dst):
    if not os.path.isdir(src):
        return 0
    n = 0
    for root, _, files in os.walk(src):
        for f in files:
            s = os.path.join(root, f); d = os.path.join(dst, os.path.relpath(s, src))
            if not os.path.exists(d):
                os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copy2(s, d); n += 1
    return n
for prev in PREV_OUTPUTS:
    n1 = restore(os.path.join(prev, "engramdelta", "results"), str(REPO / "results"))
    n2 = restore(os.path.join(prev, "ckpt"), str(WORK / "ckpt"))
    print(f"restored from {prev}: {n1} result files, {n2} checkpoint files")
if os.environ.get("HF_REPO"):
    run("python -m engramdelta.scripts.sync pull results", check=False)

# 5. hardware
if NEEDS_GPU:
    run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv", check=False)
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPUs:", torch.cuda.device_count())
NPROC = max(1, torch.cuda.device_count())
if NEEDS_GPU and NPROC < 2:
    print("!! WARNING: fewer than 2 GPUs. Set Accelerator = 'GPU T4 x2' in the notebook settings.")

## 1 · NCCL smoke test (2 processes, one per T4)

In [ ]:
rc = run("torchrun --standalone --nproc_per_node=2 -m engramdelta.scripts.nccl_smoke", check=False, timeout=300)
if rc != 0:
    print("\n!! NCCL test failed or hung. Retrying with NCCL_P2P_DISABLE=1 NCCL_IB_DISABLE=1 ...")
    fb = {"NCCL_P2P_DISABLE": "1", "NCCL_IB_DISABLE": "1", "NCCL_DEBUG": "INFO"}
    rc = run("torchrun --standalone --nproc_per_node=2 -m engramdelta.scripts.nccl_smoke", check=False, env=fb, timeout=300)
    if rc == 0:
        os.environ.update({"NCCL_P2P_DISABLE": "1", "NCCL_IB_DISABLE": "1"})
        print('>> Fallback works. In EVERY later notebook set NCCL_ENV = {"NCCL_P2P_DISABLE": "1", "NCCL_IB_DISABLE": "1"}')
    else:
        raise RuntimeError("NCCL does not work on this session: stop and investigate (NCCL_DEBUG=INFO output above).")

## 2 · FLA install check

In [ ]:
run("python -c \"import importlib.metadata as m, triton; print('fla-core', m.version('fla-core'), '| triton', triton.__version__); "
    "from fla.ops.gated_delta_rule import chunk_gated_delta_rule; print('fla.ops import OK')\"", check=False)
run("python -c \"import torch; print('compute capability', torch.cuda.get_device_capability(0))\"", check=False)

## 3 · Test 4: kernel correctness (vs FLA naive reference) and speed

In [ ]:
run("python -m engramdelta.scripts.bench_kernels --out results/phase0/E0.kernels")
latest = sorted(glob.glob("results/phase0/E0.kernels/*.json"))[-1]
kern = json.load(open(latest))
print(json.dumps(kern["verdict"], indent=1))
RECOMMENDED = kern["recommended_backend"]
print("\n>> recommended GDN_BACKEND =", RECOMMENDED)
assert all(v == "PASS" for k, v in kern["verdict"].items() if k.startswith("torch_chunk")), "torch kernel FAILED: stop and ask"

In [ ]:
# The same comparison as a pytest (includes the CUDA-only FLA test)
run("python -m pytest tests/test_kernels.py -q -rs", check=False)

## 4 · DDP throughput: 1 GPU vs 2 GPUs (synchronized DDP)
150 steps of A1 and A2 at (N=64, F=512, P=64, a), for each usable backend. The first steps include Triton
compilation, so FLA numbers are slightly pessimistic. Speedup = tokens/s at 2 GPUs ÷ at 1 GPU.
The plan's decision point: if the speedup for these tiny models is below 1.3×, report it before Phase 1.

In [ ]:
backends = ["torch"] + (["fla"] if RECOMMENDED == "fla" else [])
for be in backends:
    for arm in ("A1", "A2"):
        for nproc in (1, 2):
            rid = f"E0.thr__{arm}__{be}__np{nproc}"
            launcher = f"torchrun --standalone --nproc_per_node={nproc}" if nproc > 1 else "python"
            run(f"{launcher} -m engramdelta.scripts.train --config configs/phase1/{arm}.yaml --run-id {rid} "
                f"--set phase=phase0 --set group=E0.thr --set data.N=64 --set data.F=512 "
                f"--set train.steps=150 --set train.log_every=50 --set data.eval_size=400",
                env={"ENGRAMDELTA_GDN_BACKEND": be}, check=False)
import pandas as pd
rows = []
for f in sorted(glob.glob("results/phase0/E0.thr/*.json")):
    r = json.load(open(f))
    rows.append({"run": r["run_id"], "arm": r["arm"], "backend": r["gdn_backend"], "world": r["world_size"],
                 "tok/s": round(r["throughput_tokens_per_s"]), "sec_per_step": r["train_seconds"] / r["train"]["steps"]})
t = pd.DataFrame(rows); print(t.to_string(index=False))
for (arm, be), g in t.groupby(["arm", "backend"]):
    if set(g["world"]) == {1, 2}:
        sp = g[g.world == 2]["tok/s"].iloc[0] / g[g.world == 1]["tok/s"].iloc[0]
        print(f"DDP speedup {arm} [{be}]: {sp:.2f}x")

## Decision (write it into PREREG.md → "Phase 0 outcomes")
* `GDN_BACKEND` for all later notebooks = the recommended value above (keep `FLA_SPEC` as is).
* If NCCL needed the fallback, set `NCCL_ENV` in all later notebooks.
* Tokens/s at 2 GPUs (torch / fla) feed the Phase 1 budget in NB02.

In [ ]:
# ===================== End of session: persist =====================
if os.environ.get("HF_REPO"):
    run("python -m engramdelta.scripts.sync push results", check=False)
n = sum(len(f) for _, _, f in os.walk(REPO / "results"))
print(f"{n} files under results/. They are part of this notebook's output: Save Version keeps them.")
print("Next session: attach this notebook's output as an input and add its path to PREV_OUTPUTS (or use HF_REPO).")